In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql.window import Window
import uuid


In [0]:
%sql
use catalog novacart_workspace;
create schema if not exists silver_schema;



In [0]:
silver_run_id=str(uuid.uuid4())
print("current Silver Run ID: ", silver_run_id)


In [0]:
%sql
create table if not exists silver_schema.progressing_control(
    layer string,
    entity_name string,
    last_processed_bronze_run_id string,
    last_processed_bronze_ingested_at timestamp,
    rows_merged bigint,
    run_status string,
    silver_run_id string,
    updated_at timestamp
)
using delta;

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        dt.alias("target").merge(df_source.alias("source"),f"target.{join_key}=source.{join_key}").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        df_source.write.format("delta").saveAsTable(target_table)



In [0]:
def get_last_processed_bronze_ingested_at(entity_name):
    ctrl=spark.table("novacart_workspace.silver_schema.progressing_control").filter((col('layer')=='silver') &
    (col("entity_name") == entity_name) & 
    (col("run_status")== "SUCCESS")).orderBy(col("updated_at").desc()).limit(1)
    rows = ctrl.collect() 
    if not rows:
        return None 
    else:
        return rows[0]['last_processed_bronze_ingested_at']


In [0]:
def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    ctrl_df=spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            rows_merged,
            "SUCCESS",
            silver_run_id,
            datetime.utcnow()
        )
        ],
        schema=""" layer string,
                    entity_name string,
                    last_processed_bronze_run_id string,
                    last_processed_bronze_ingested_at timestamp,
                    rows_merged bigint,
                    run_status string,
                    silver_run_id string,
                    updated_at timestamp  """
    )
    df=DeltaTable.forName(spark,"novacart_workspace.silver_schema.progressing_control")
    df.alias("t").merge(ctrl_df.alias("s"),"t.layer=s.layer and t.entity_name=s.entity_name").whenMatchedUpdate(set={
        "last_processed_bronze_run_id":"s.last_processed_bronze_run_id",
        "last_processed_bronze_ingested_at":"s.last_processed_bronze_ingested_at",
        "rows_merged":"s.rows_merged",
        "run_status":"s.run_status",
        "silver_run_id":"s.silver_run_id",
        "updated_at":"s.updated_at"
    }).whenNotMatchedInsertAll().execute()



In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    else:
        return bronze_df.filter(col("bronze_ingested_at") > last_ingested_at),last_ingested_at


In [0]:
%sql
select * from novacart_workspace.bronze_schema.orders_raw

In [0]:
orders_inc, last_orders_ingested_at=get_incremental_bronze("novacart_workspace.bronze_schema.orders_raw","orders")

orders_inc_count = orders_inc.count()
print("orders rows to process in silver: ", orders_inc_count)

if orders_inc_count >0:
    order_window = (
        Window.partitionBy("order_id")
        .orderBy(
            col("updated_at").cast("timestamp").desc(),
            col("bronze_ingested_at").desc()
        )
    )

    order_cleaned = (
        orders_inc
        .withColumn(
            "order_status",
            upper(trim(col("order_status")))
        )
        .withColumn(
            "order_status",
            when(
                col("order_status") == "",
                lit(None)
            ).otherwise(col("order_status"))
        )
        .withColumn(
            "order_amount",
            regexp_replace(col("order_amount"), r"[$, ]", "")
        )
        .withColumn(
            "order_amount",
            when(
                trim(col("order_amount")).isin("N/A", "NULL", "??", ""),
                lit(None)
            ).otherwise(col("order_amount"))
        )
        .withColumn(
            "order_amount",
            col("order_amount").cast("double")
        )
        .withColumn("created_at", to_timestamp("created_at"))
        .withColumn("updated_at", to_timestamp("updated_at"))
        .withColumn("row_rank", row_number().over(order_window))
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )


    upsert_to_silver(order_cleaned,"novacart_workspace.silver_schema.orders_cleaned","order_id")

    orders_validated = (
    order_cleaned
    .withColumn(
        "to_be_verified_by_orders_team",
        when(col("customer_id").isNull(), "verify_customer_id")
        .when(col("product_id").isNull(), "verify_product_id")
        .when(
            col("order_status").isNull() |
            (trim(col("order_status")) == ""),
            "verify_order_status"
        )
        .when(
            col("order_amount").isNull() |
            (col("order_amount") < 0),
            "verify_order_amount"
        )
        .otherwise("No Issues")
    )
    .withColumn(
        "check_order_amount",
        when(
            col("order_amount").isNull() |
            (col("order_amount") < 0),
            lit(True)
        ).otherwise(lit(False))
    )
    .withColumn("order_date", to_date("created_at"))
    .withColumn("order_year", year("created_at"))
    .withColumn("order_month", month("created_at"))
    .withColumn("order_day", dayofmonth("created_at"))
    .withColumn("order_dow", date_format(col("created_at"), "E")))

    orders_good=orders_validated.filter(col("to_be_verified_by_orders_team") == "No Issues")

    orders_bad=(orders_validated.filter(col("to_be_verified_by_orders_team")!="No Issues").withColumn("quarantine_ts",current_timestamp()))

    upsert_to_silver(orders_good,"novacart_workspace.silver_schema.orders_transformed","order_id")

    orders_bad.write.format('delta').mode('append').saveAsTable("novacart_workspace.silver_schema.orders_quarantine")


    mx_ingested = orders_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run=(orders_inc.filter(col("bronze_ingested_at") == lit(mx_ingested)).agg(max("bronze_run_id").alias("mx")).collect()[0]["mx"])


    upsert_silver_control("orders",mx_run,mx_ingested,orders_good.count())
else:
    print("No new orders to process for silver")

    upsert_silver_control("orders",None,last_orders_ingested_at,orders_inc_count)


                                                                







In [0]:

product_inc, last_product_ingested_at = get_incremental_bronze(
    "novacart_workspace.bronze_schema.products_raw",
    "products"
)

product_inc_count = product_inc.count()
print("Products rows to process in Silver:", product_inc_count)

if product_inc_count > 0:

    product_window = (
        Window.partitionBy("product_id")
        .orderBy(
            col("updated_at").cast("timestamp").desc(),
            col("bronze_ingested_at").desc()
        )
    )

    # Clean and standardize product records.
    products_cleaned = (
        product_inc
        .withColumn(
            "product_name",
            upper(trim(col("product_name")))
        )
    .withColumn(
    "product_name",
    regexp_replace(col("product_name"), r"[-_]", " ")
)
        .withColumn(
            "product_name",
            when(col("product_name") == "", lit(None))
            .otherwise(col("product_name"))
        )
        .withColumn(
            "category",
            when(
                upper(trim(col("category"))).contains("ELECTRONICS"),
                "ELECTRONICS"
            )
            # .when(upper(trim(col("category"))) == "FITNESS", "FITNESS")
            # .when(upper(trim(col("category"))) == "LIFESTYLE", "LIFESTYLE")
            .otherwise(upper(trim(col("category"))))
        )
        .withColumn("price", trim(col("price")))
        .withColumn("price", regexp_replace(col("price"), r"\$", ""))
        .withColumn("price", regexp_replace(col("price"), ",", ""))
        .withColumn("price", regexp_replace(col("price"), r"\s+", ""))
        .withColumn("price", expr("try_cast(price as double)"))
        .withColumn("updated_at", to_timestamp("updated_at"))
        .withColumn("row_rank", row_number().over(product_window))
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Save cleaned products to Silver.
    upsert_to_silver(
        products_cleaned,
        "novacart_workspace.silver_schema.products_cleaned",
        "product_id"
    )

    # Apply data-quality rules.
    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
            when(col("product_name").isNull(), "verify_product_name")
            .when(col("category").isNull(), "verify_category")
            .when(
                col("price").isNull() | (col("price") <= 0),
                "verify_price"
            )
            .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            when(
                col("price").isNull() | (col("price") <= 0),
                "invalid_price"
            ).otherwise("valid_price")
        )
    )

    # Separate valid and invalid products.
    product_good = products_validated.filter(
        (col("to_be_verified_by_products_team") == "No Issues") &
        (col("check_product_price") == "valid_price")
    )

    product_bad = (
        products_validated
        .filter(
            (col("to_be_verified_by_products_team") != "No Issues") |
            (col("check_product_price") != "valid_price")
        )
        .withColumn("quarantine_ts", current_timestamp())
    )

    # Upsert valid products into the transformed Silver table.
    upsert_to_silver(
        product_good,
        "novacart_workspace.silver_schema.products_transformed",
        "product_id"
    )

    # Append invalid products to quarantine.
    product_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_workspace.silver_schema.products_quarantine"
    )

    # Find the latest Bronze ingestion timestamp.
    mx_ingested = (
        product_inc
        .agg(max("bronze_ingested_at").alias("mx"))
        .collect()[0]["mx"]
    )

    # Find the Bronze run ID associated with that timestamp.
    latest_run_rows = (
        product_inc
        .filter(col("bronze_ingested_at") == lit(mx_ingested))
        .select("bronze_run_id")
        .distinct()
        .limit(2)
        .collect()
    )

    if len(latest_run_rows) == 1:
        mx_run = latest_run_rows[0]["bronze_run_id"]
    else:
        raise ValueError(
            "Latest ingestion timestamp does not identify exactly one Bronze run ID."
        )

    # Update the Silver control table after processing.
    upsert_silver_control(
        "products",
        mx_run,
        mx_ingested,
        product_good.count()
    )

else:
    print("No new products to process for Silver")

    upsert_silver_control(
        "products",
        None,
        last_product_ingested_at,
        product_inc_count
        
    )


In [0]:

payments_inc, last_payments_ingested_at = get_incremental_bronze(
    "novacart_workspace.bronze_schema.payments_raw",
    "payments"
)

payments_inc_count = payments_inc.count()
print("Payments rows to process in Silver:", payments_inc_count)

if payments_inc_count > 0:

    payment_window = (
        Window.partitionBy("payment_id")
        .orderBy(
            col("processed_at").cast("timestamp").desc(),
            col("bronze_ingested_at").desc()
        )
    )

    # Clean and deduplicate payment records.
    payments_cleaned = (
        payments_inc
        .withColumn(
            "payment_status",
            upper(trim(col("payment_status")))
        )
        .withColumn(
            "payment_status",
            when(
                col("payment_status") == "",
                lit(None)
            ).otherwise(col("payment_status"))
        )
        .withColumn("paid_amount", trim(col("paid_amount")))
        .withColumn(
            "paid_amount",
            regexp_replace(col("paid_amount"), r"\$", "")
        )
        .withColumn(
            "paid_amount",
            regexp_replace(col("paid_amount"), ",", "")
        )
        .withColumn(
            "paid_amount",
            regexp_replace(col("paid_amount"), r"\s+", "")
        )
        .withColumn(
            "paid_amount",
            expr("try_cast(paid_amount as double)")
        )
        .withColumn("processed_at", to_timestamp("processed_at"))
        .withColumn("row_rank", row_number().over(payment_window))
        .filter(col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", lit(silver_run_id))
    )

    # Save cleaned payments.
    upsert_to_silver(
        payments_cleaned,
        "novacart_workspace.silver_schema.payments_cleaned",
        "payment_id"
    )

    # Apply data-quality rules.
    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
            when(col("order_id").isNull(), "verify_order_id")
            .when(
                col("payment_status").isNull(),
                "verify_payment_status"
            )
            .when(
                col("paid_amount").isNull() |
                (col("paid_amount") <= 0),
                "verify_paid_amount"
            )
            .otherwise("No Issues")
        )
        .withColumn(
            "check_paid_amount",
            when(
                col("paid_amount").isNull() |
                (col("paid_amount") <= 0),
                lit(True)
            ).otherwise(lit(False))
        )
    )

    # Separate valid and invalid payments.
    payment_good = payments_validated.filter(
        col("to_be_verified_by_payments_team") == "No Issues"
    )

    payment_bad = (
        payments_validated
        .filter(
            col("to_be_verified_by_payments_team") != "No Issues"
        )
        .withColumn("quarantine_ts", current_timestamp())
    )

    # Write valid payments to the transformed Silver table.
    upsert_to_silver(
        payment_good,
        "novacart_workspace.silver_schema.payments_transformed",
        "payment_id"
    )

    # Append invalid payments to quarantine.
    payment_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_workspace.silver_schema.payments_quarantine"
    )

    # Find the latest Bronze ingestion timestamp.
    mx_ingested = (
        payments_inc
        .agg(max("bronze_ingested_at").alias("mx"))
        .first()["mx"]
    )

    # Find the Bronze run ID associated with that timestamp.
    latest_run_rows = (
        payments_inc
        .filter(col("bronze_ingested_at") == lit(mx_ingested))
        .select("bronze_run_id")
        .distinct()
        .limit(2)
        .collect()
    )

    if len(latest_run_rows) == 1:
        mx_run = latest_run_rows[0]["bronze_run_id"]
    else:
        raise ValueError(
            "Latest ingestion timestamp does not identify exactly one Bronze run ID."
        )

    # Update the control table after processing succeeds.
    upsert_silver_control(
        "payments",
        mx_run,
        mx_ingested,
        payment_good.count()
    )

else:
    print("No new payments to process for Silver")

    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        0
    )


In [0]:
%sql
select * from silver_schema.payments_quarantine